In [6]:
from openmm.app import *
from openmm import *
import openmm.unit as unit
import openmm.app.metadynamics as mtd
import matplotlib.pyplot as plt 
from sys import stdout

### Setting up OMM system.

In [7]:
pdb = PDBFile('/home/ethanz/data_boltz_likelihood/md/chignolin_folded.pdb') # Hydrogens already added.
forcefield = ForceField('amber19-all.xml', 'implicit/gbn2.xml')
system = forcefield.createSystem(
    pdb.topology, 
    nonbondedMethod=NoCutoff, 
    nonbondedCutoff=1 * unit.nanometer, 
    constraints=HBonds
)

### Unbiased simulation.

In [5]:
# Minimization.
temp_omm = 300 * unit.kelvin
integrator = LangevinIntegrator(
    temp_omm,
    1.0 / unit.picosecond,
    2.0 * unit.femtoseconds
)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.getPositions())
tolerance = 0.1*unit.kilojoules_per_mole/unit.angstroms
simulation.minimizeEnergy(tolerance=tolerance,maxIterations=10000)
print('Finished with energy minimization.')

# Equilibration.
simulation.context.setVelocitiesToTemperature(temp_omm) # Only set velocities after minimization. 
simulation.reporters.append(StateDataReporter('equilibrate.log', 1000, step=True, temperature=True, potentialEnergy=True, totalEnergy=True, speed=True))
simulation.step(10000)
positions = simulation.context.getState(getPositions=True).getPositions()
simulation.saveCheckpoint('min_eq.chk')
print('Done equilibrating, saved checkpoint.')
# update the current context with changes in system
# simulation.context.reinitialize()

# Production run.
integrator = LangevinIntegrator(
    temp_omm,
    1.0 / unit.picosecond,
    2.0 * unit.femtoseconds
)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(positions)
simulation.context.setVelocitiesToTemperature(temp_omm)

# Set simulation reporters.
simulation.reporters.append(DCDReporter('mtd_5UG9.dcd', 5000))
simulation.reporters.append(StateDataReporter('mtd_5UG9.out', 5000, step=True, 
    potentialEnergy=True, temperature=True, progress=True, remainingTime=True, 
    speed=True, totalSteps=1e8, separator='\t'))

simulation.step(1e8) # 2fs * 1e8 = 200 ns.

Finished with energy minimization.
Done equilibrating, saved checkpoint.


KeyboardInterrupt: 

### MTD simulation.

In [9]:
# Preparing bias.
# First, prepare the custom CV force. 
ca_indices = [atom.index for atom in pdb.topology.atoms() if atom.name == "CA"]
i, j = ca_indices[0], ca_indices[-1]

cv_force = CustomCVForce("e2e_dist")
cv_force.addCollectiveVariable("e2e_dist", CustomBondForce("r"))
cv_force.getCollectiveVariable(0).addBond(i, j)
bias_variable = mtd.BiasVariable(
    force=cv_force,
    minValue=0.0,
    maxValue=2.0, 
    biasWidth=0.1,
    periodic=False
)

# Running simulation
# Minimization - removing steric clashes from hydrogens.
temp_omm = 300 * unit.kelvin
integrator = LangevinIntegrator(
    temp_omm,
    1.0 / unit.picosecond,
    2.0 * unit.femtoseconds
)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.getPositions())
tolerance = 0.1*unit.kilojoules_per_mole/unit.angstroms
simulation.minimizeEnergy(tolerance=tolerance,maxIterations=10000)
print('Finished with energy minimization.')

# Equilibration.
simulation.context.setVelocitiesToTemperature(temp_omm) # Only set velocities after minimization. 
simulation.reporters.append(StateDataReporter('relax-hydrogens.log', 1000, step=True, temperature=True, potentialEnergy=True, totalEnergy=True, speed=True))
simulation.step(10000)
positions = simulation.context.getState(getPositions=True).getPositions()
simulation.saveCheckpoint('min_eq.chk')
print('Done equilibrating, saved checkpoint.')
# update the current context with changes in system
simulation.context.reinitialize()

# Creating Metadynamics object.
# Requires a System and list of BiasViariables. 
integrator = LangevinIntegrator(
    temp_omm,
    1.0 / unit.picosecond,
    2.0 * unit.femtoseconds
)
BIAS_DIR = '/home/ethanz/boltz-likelihoods/SPLASH/chignolin_bias'
temp_omm = 300 * unit.kelvin
os.makedirs(BIAS_DIR, exist_ok=True)
meta = mtd.Metadynamics(
    system=system,
    variables=[bias_variable],
    temperature=temp_omm,
    biasFactor=5.0,
    height=1.0 * unit.kilojoule_per_mole,
    frequency=500,
    saveFrequency=500,
    biasDir=BIAS_DIR
)

simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(positions)
simulation.context.setVelocitiesToTemperature(temp_omm)
simulation.step(100)
print("100 step sanity check done.")

# Set simulation reporters.
simulation.reporters.append(DCDReporter('mtd_5UG9.dcd', 5000))
simulation.reporters.append(StateDataReporter('mtd_5UG9.out', 5000, step=True, 
    potentialEnergy=True, temperature=True, progress=True, remainingTime=True, 
    speed=True, totalSteps=10000, separator='\t'))

# Run small-scale simulation (10ns, 5*10^6 steps) and plot the free energy landscape
meta.step(simulation, 10000)

Finished with energy minimization.
Done equilibrating, saved checkpoint.
100 step sanity check done.


In [11]:
gridWidth = bias_variable.gridWidth
free_energies = meta.getFreeEnergy()
import numpy as np

print(type(gridWidth), type(free_energies))
print(np.array(free_energies))

# The i’th position along an axis corresponds to minValue + i*(maxValue-minValue)/gridWidth.
# lengths = [bias_variable.minValue + i*(bias_variable.maxValue-bias_variable.minValue)/gridWidth for i in range(gridWidth)]
# plt.plot(lengths, free_energies)
# plt.xlabel("End-to-end distance (nm)")
# plt.ylabel("Free energy (kJ/mol)")
# plt.title("Free energy landscape of chignolin folding")
# plt.show()

<class 'int'> <class 'openmm.unit.quantity.Quantity'>
[-1.12447266e-02 -2.41746564e-02 -5.00324470e-02 -9.97018313e-02
 -1.91338620e-01 -3.53710386e-01 -6.30009408e-01 -1.08148678e+00
 -1.78980560e+00 -2.85662753e+00 -4.39883163e+00 -6.53813545e+00
 -9.38488395e+00 -1.30173409e+01 -1.74596420e+01 -2.26630895e+01
 -2.84960292e+01 -3.47466521e+01 -4.11406322e+01 -4.73720163e+01
 -5.31422087e+01 -5.81993791e+01 -6.23700918e+01 -6.55766896e+01
 -6.78374996e+01 -6.92511597e+01 -6.99700075e+01 -7.01694634e+01
 -7.00202666e+01 -6.96685455e+01 -6.92258461e+01 -6.87683958e+01
 -6.83428698e+01 -6.79751468e+01 -6.76789111e+01 -6.74620596e+01
 -6.73301578e+01 -6.72872253e+01 -6.73347098e+01 -6.74696462e+01
 -6.76828626e+01 -6.79578565e+01 -6.82707350e+01 -6.85914123e+01
 -6.88860459e+01 -6.91204596e+01 -6.92640534e+01 -6.92935177e+01
 -6.91956274e+01 -6.89685319e+01 -6.86212682e+01 -6.81716174e+01
 -6.76427937e+01 -6.70596895e+01 -6.64454464e+01 -6.58189916e+01
 -6.51939110e+01 -6.45787156e+01 -6.

In [1]:
# Saving trajectory as a pdb.
import mdtraj as md
mdt_top = md.load('/home/ethanz/data_boltz_likelihood/md/chignolin.pdb').topology
mdt_traj = md.load('mtd_5UG9.dcd', top=mdt_top)
mdt_traj.save('/home/ethanz/boltz-likelihoods/SPLASH/chignolin_mtd.pdb')

dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)


In [19]:
print(simulation.context.getPlatform().getName())

CUDA
